# AGender - Step 4: the two churn tables (CPU)

The audit of Step 2B and the release selection of Step 3, on two more tables. The partner's task on both is churn.

- **Iranian Churn** (UCI 563, CC BY 4.0, real customers): is the age group (five groups) revealed by usage, and which
  attributes can be released for churn models while it stays hidden? `Age` and `Customer Value` carry the age group
  and are dropped; exact copies are removed (see `docs/step4_churn.md`).
- **IBM Telco** (IBM sample data, simulated customers): gender is simulated, so the audit should find **no** leak.
  This is the negative control: only the audit is run here, because a release cannot lower a leak that is not there.

Same four parts (train 60 %, validation 10 %, calibration 10 %, test 20 %) and five seeds as LAT.

**Kaggle settings:** Accelerator = **None** (CPU). **Internet = On** (both tables are downloaded).
Without internet, attach the two files instead (`iranian_churn.zip` or `Customer Churn.csv`, and `Telco-Customer-Churn.csv`).
**Inputs to add:** `agender-code`.
**Expected run time:** about 30-40 minutes.


In [ ]:
import os
import sys
from pathlib import Path

# The AGender code is attached as a Kaggle dataset. Find the `agender` package
# wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('audit.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The agender-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> agender-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if package.parent.name == 'agender' and (package.parent / '__init__.py').exists():
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder; link the files under the name `agender`.
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'agender').exists():
        os.symlink(package.parent, code_root / 'agender')

sys.path.insert(0, str(code_root))
import agender
print('AGender code loaded from', Path(agender.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/agender_outputs/step4')
TIME_BUDGET_MINUTES = 240
REPEATS = range(5)
EXACT_CHECK = True


In [ ]:
import json

import pandas as pd

from agender import churn, environment, step4
from agender.runner import StepRunner

OUTPUT.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {'environment': runner.run('measure the machine', environment.collect)}


def get_tables():
    folder = OUTPUT / 'data'
    records = []
    iranian = find_file('iranian_churn.zip', INPUT) or find_file('Customer Churn.csv', INPUT)
    if iranian is None:
        records.append(churn.fetch_iranian(folder))
        iranian = folder / 'iranian_churn.zip'
    telco = find_file('Telco-Customer-Churn.csv', INPUT) or find_file('telco_customer_churn.csv', INPUT)
    if telco is None:
        records.append(churn.fetch_telco(folder))
        telco = folder / 'telco_customer_churn.csv'
    raw = {'iranian': churn.read_iranian(iranian), 'telco': pd.read_csv(telco)}
    return raw, records


got = runner.run('get the two tables', get_tables)
if got is None:
    raise RuntimeError('Could not get the two tables. Turn Internet on (Settings -> Internet), or attach '
                       'iranian_churn.zip and Telco-Customer-Churn.csv as inputs, then Save & Run All again.')
raw, records = got
tables = {'iranian': churn.clean_iranian(raw['iranian']), 'telco': churn.clean_telco(raw['telco'])}
summary['data'] = {'downloads': records,
                   'iranian': churn.describe('iranian', raw['iranian'], tables['iranian'], 'AGE',
                                             churn.iranian_attributes(tables['iranian'])),
                   'telco': churn.describe('telco', raw['telco'], tables['telco'], 'FEMALE',
                                           churn.telco_attributes(tables['telco']))}
print(json.dumps(summary['data'], indent=1))
table_splits = {name: step4.prepare(name, table, OUTPUT) for name, table in tables.items()}


In [ ]:
for name in ('iranian', 'telco'):
    summary[f'{name} audit'] = runner.run(f'{name}: audit', step4.audit, name, tables[name], table_splits[name],
                                          OUTPUT, repeats=REPEATS)
    print(json.dumps(summary[f'{name} audit'], indent=1))


In [ ]:
summary['iranian release'] = runner.run('iranian: release selection', step4.release, 'iranian', tables['iranian'],
                                       table_splits['iranian'], OUTPUT, repeats=REPEATS, exact=EXACT_CHECK)
print(json.dumps(summary['iranian release'], indent=1))


In [ ]:
summary['steps'] = runner.steps
step4.save_summary(summary, OUTPUT / 'step4_summary.json')
print('Failed or skipped steps:', runner.failed() or 'none')
